# BPE and a Transformer encoder from first principles

This notebook implements byte-pair encoding over a toy corpus and builds a compact Transformer encoder in PyTorch. The goal is to expose the mechanics hidden by high-level libraries: merge statistics, padding masks, residual connections and tensor dimensions.

In [2]:
from collections import Counter
import re

corpus = ["low lower lowest", "newer wider", "low new"]
vocabulary = Counter()
for sentence in corpus:
    for word in sentence.split():
        vocabulary[tuple(word) + ("</w>",)] += 1

def pair_counts(vocab):
    counts = Counter()
    for symbols, frequency in vocab.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += frequency
    return counts

def merge_pair(vocab, pair):
    merged = Counter()
    for symbols, frequency in vocab.items():
        output, index = [], 0
        while index < len(symbols):
            if index + 1 < len(symbols) and symbols[index:index+2] == pair:
                output.append("".join(pair))
                index += 2
            else:
                output.append(symbols[index])
                index += 1
        merged[tuple(output)] += frequency
    return merged

merges = []
for _ in range(12):
    counts = pair_counts(vocabulary)
    if not counts:
        break
    best = counts.most_common(1)[0][0]
    merges.append(best)
    vocabulary = merge_pair(vocabulary, best)
print("learned merges:", merges)
print("final vocabulary:", vocabulary)

learned merges: [('l', 'o'), ('lo', 'w'), ('e', 'r'), ('er', '</w>'), ('low', '</w>'), ('n', 'e'), ('ne', 'w'), ('low', 'er</w>'), ('low', 'e'), ('lowe', 's'), ('lowes', 't'), ('lowest', '</w>')]
final vocabulary: Counter({('low</w>',): 2, ('lower</w>',): 1, ('lowest</w>',): 1, ('new', 'er</w>'): 1, ('w', 'i', 'd', 'er</w>'): 1, ('new', '</w>'): 1})


BPE repeatedly merges the most frequent adjacent symbol pair. Frequent words become short token sequences, while unseen words remain representable through smaller units. The merge table, normalization and pre-tokenizer together define the tokenizer and must be versioned with the model.

In [4]:
def encode_word(word, learned_merges):
    symbols = tuple(word) + ("</w>",)
    for pair in learned_merges:
        symbols = next(iter(merge_pair(Counter({symbols: 1}), pair)))
    return symbols

for word in ["low", "lowest", "newest"]:
    print(word, "->", encode_word(word, merges))

low -> ('low</w>',)
lowest -> ('lowest</w>',)
newest -> ('new', 'e', 's', 't', '</w>')


## Minimal Transformer encoder

`key_padding_mask=True` marks padding positions that keys and values must ignore. Positional embeddings are required because self-attention alone is permutation equivariant.

In [6]:
import torch
from torch import nn

class TinyTransformerEncoder(nn.Module):
    def __init__(self, vocab_size, model_dim=64, heads=4, layers=2, max_length=128):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, model_dim, padding_idx=0)
        self.position_embedding = nn.Embedding(max_length, model_dim)
        block = nn.TransformerEncoderLayer(
            d_model=model_dim, nhead=heads, dim_feedforward=4 * model_dim,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(block, num_layers=layers)
        self.classifier = nn.Linear(model_dim, 2)

    def forward(self, input_ids, attention_mask):
        positions = torch.arange(input_ids.shape[1], device=input_ids.device)
        hidden = self.token_embedding(input_ids) + self.position_embedding(positions)[None]
        hidden = self.encoder(hidden, src_key_padding_mask=~attention_mask.bool())
        pooled = (hidden * attention_mask[..., None]).sum(1) / attention_mask.sum(1, keepdim=True)
        return self.classifier(pooled)

network = TinyTransformerEncoder(vocab_size=100)
input_ids = torch.tensor([[3, 8, 9, 0], [7, 4, 2, 1]])
attention_mask = input_ids.ne(0)
logits = network(input_ids, attention_mask)
print("logits shape:", logits.shape)
assert logits.shape == (2, 2)

logits shape: torch.Size([2, 2])


nlp:cell_6:13: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


In [7]:
parameter_count = sum(parameter.numel() for parameter in network.parameters())
trainable_count = sum(parameter.numel() for parameter in network.parameters() if parameter.requires_grad)
print(f"parameters={parameter_count:,}; trainable={trainable_count:,}")

loss = nn.CrossEntropyLoss()(logits, torch.tensor([0, 1]))
loss.backward()
assert all(parameter.grad is not None for parameter in network.parameters() if parameter.requires_grad)
print("forward and backward passes completed; loss=", round(loss.item(), 4))

parameters=114,690; trainable=114,690
forward and backward passes completed; loss= 0.5504


The implementation is intentionally small and omits checkpointing, mixed precision and distributed training. It is suitable for testing architectural understanding. Production pretraining additionally requires a streaming data pipeline, a causal or masked-language objective, learning-rate scheduling, checkpoint recovery and evaluation for memorization and data contamination.